# 04 — Phase 2 multi-year residual refinement

Phase 2 freezes the Phase 1 Attention U-Net and optimises only the Conv3D residual head on ordered four-year inputs. The notebook retains the executed residual-only and optional pseudo-target regularisation variants described in the manuscript.

In [ ]:
from __future__ import annotations

import hashlib
import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT = Path(r"C:\Users\Dell\Desktop\Publication_Clarck\Natural_Sampling")
CONFIG_PATH = PROJECT / "Config" / "pipeline_config.json"
SOURCE_ROOT = PROJECT / "Source" / "Project"
assert PROJECT.is_dir(), PROJECT
assert CONFIG_PATH.is_file(), CONFIG_PATH
PIPELINE = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))

def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

print("Projet :", PROJECT)
print("Configuration :", CONFIG_PATH)
print("Python :", sys.executable)


## Sequence construction, model configuration, and training

Complete same-month four-year sequences are built from the fixed spatial partitions. The code records the production configuration and checkpoint lineage for each landscape.

In [ ]:
REGISTRY_PATH = SOURCE_ROOT / "final_selected_phase2_models.json"
assert REGISTRY_PATH.is_file(), REGISTRY_PATH
FINAL_REGISTRY = json.loads(REGISTRY_PATH.read_text(encoding="utf-8"))
assert FINAL_REGISTRY.get("test_used_for_selection") is False

SITE_METADATA = {
    "ifran": {"forest": "Ifran", "domain": (2.0, 45.0)},
    "maamoura": {"forest": "Maamoura", "domain": (2.0, 20.0)},
    "agadir": {"forest": "Agadir", "domain": (2.0, 20.0)},
}
FINAL_SPECS = {}
for site, metadata in SITE_METADATA.items():
    selected = FINAL_REGISTRY["models"][site]
    checkpoint = Path(selected["checkpoint"])
    if not checkpoint.is_file():
        raise FileNotFoundError(checkpoint)
    if sha256(checkpoint) != selected["checkpoint_sha256"]:
        raise RuntimeError(f"{site}: final selected checkpoint hash mismatch")
    FINAL_SPECS[site] = {
        **metadata,
        "product": selected["product"],
        "D": float(selected["drop_m"]),
        "K": int(selected["K"]),
        "lambda_growth": float(selected["lambda_temp"]),
        "huber_delta": float(selected["huber_delta"]),
        "checkpoint": str(checkpoint),
        "checkpoint_sha256": selected["checkpoint_sha256"],
    }

pd.DataFrame(FINAL_SPECS).T[
    ["forest", "product", "D", "K", "lambda_growth", "huber_delta", "checkpoint_sha256", "domain"]
]


In [ ]:
PHASE2_RUNNER = SOURCE_ROOT / "final_phase2_harmonized_workflow.py"
assert PHASE2_RUNNER.is_file(), PHASE2_RUNNER
PHASE2_PYTHON = Path(r"C:\Users\Dell\Desktop\Article_Maroc_Agadir\Env_Workspace_agadir\.venv310\Scripts\python.exe")
if not PHASE2_PYTHON.is_file():
    PHASE2_PYTHON = Path(sys.executable)

# Read-only preflight: prints raw versus retained T4 record counts for every split.
subprocess.run(
    [str(PHASE2_PYTHON), "-u", str(PHASE2_RUNNER), "--forest", "all", "--preflight"],
    check=True,
)

In [ ]:
# New isolated training: no existing checkpoint or result is overwritten.
import os
import queue
import threading
import time

RETRAIN_HARMONIZED_PHASE2 = False  # final checkpoints are frozen; never retrain here
EVALUATE_SELECTED_TEST = True
SITES_TO_RUN = ("ifran", "maamoura", "agadir")
HEARTBEAT_SECONDS = 30
LIVE_LOG_ROOT = PROJECT / "Logs" / "Phase2_Harmonized_GEDIAnchored_NaturalP1"
LIVE_LOG_ROOT.mkdir(parents=True, exist_ok=True)

def run_live(command, log_path, heartbeat_seconds=30):
    # Stream child stdout/stderr into Jupyter and a persistent UTF-8 log.
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["PYTHONIOENCODING"] = "utf-8"
    env["PYTHONUTF8"] = "1"
    process = subprocess.Popen(
        command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding="utf-8", errors="replace", bufsize=1, env=env,
    )
    messages = queue.Queue()
    def reader():
        try:
            for line in process.stdout: messages.put(line)
        finally: messages.put(None)
    threading.Thread(target=reader, daemon=True).start()
    started = last_message = time.monotonic(); stream_closed = False
    with Path(log_path).open("a", encoding="utf-8") as log:
        log.write("\nCOMMAND: " + subprocess.list2cmdline(command) + "\n"); log.flush()
        while not stream_closed:
            try: message = messages.get(timeout=1.0)
            except queue.Empty:
                now = time.monotonic()
                if now - last_message >= heartbeat_seconds:
                    heartbeat = f"[RUNNING] elapsed={(now-started)/60:.1f} min | PID={process.pid} | waiting for next message..."
                    print(heartbeat, flush=True); log.write(heartbeat + "\n"); log.flush(); last_message = now
                continue
            if message is None:
                stream_closed = True; continue
            print(message, end="", flush=True); log.write(message); log.flush(); last_message = time.monotonic()
    code = process.wait()
    print(f"[PROCESS EXIT] code={code} | log={log_path}", flush=True)
    if code: raise subprocess.CalledProcessError(code, command)

print("[FINAL SELECTED PHASE 2]", "EVALUATION ONLY" if EVALUATE_SELECTED_TEST else "PREFLIGHT ONLY", flush=True)
for site in SITES_TO_RUN:
    command = [str(PHASE2_PYTHON), "-u", str(PHASE2_RUNNER), "--forest", site, "--preflight"]
    if RETRAIN_HARMONIZED_PHASE2: command.append("--train")
    if EVALUATE_SELECTED_TEST: command.append("--evaluate-test")
    log_path = LIVE_LOG_ROOT / f"{site}_console_stream.log"
    print("\n=====", site.upper(), "=====", flush=True)
    print(subprocess.list2cmdline(command), flush=True)
    print("Live log:", log_path, flush=True)
    run_live(command, log_path, heartbeat_seconds=HEARTBEAT_SECONDS)

In [ ]:
def _column(frame: pd.DataFrame, candidates: tuple[str, ...]) -> str:
    for name in candidates:
        if name in frame.columns:
            return name
    raise KeyError(f"Aucune colonne parmi {candidates}; colonnes={list(frame.columns)}")

def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true, y_pred = y_true[mask], y_pred[mask]
    residual = y_pred - y_true
    slope, intercept = np.polyfit(y_true, y_pred, 1)
    corr = np.corrcoef(y_true, y_pred)[0, 1]
    rmse = float(np.sqrt(np.mean(residual ** 2)))
    r2 = float(1.0 - np.sum(residual ** 2) / np.sum((y_true - y_true.mean()) ** 2))
    std_ratio = float(np.std(y_pred) / np.std(y_true))
    beta = float(y_pred.mean() / y_true.mean())
    kge = float(1.0 - np.sqrt((corr - 1.0) ** 2 + (std_ratio - 1.0) ** 2 + (beta - 1.0) ** 2))
    return {
        "n": len(y_true), "r2": r2, "mae": float(np.mean(np.abs(residual))),
        "rmse": rmse, "bias": float(residual.mean()), "slope": float(slope),
        "intercept": float(intercept), "corr": float(corr),
        "std_ratio": std_ratio, "kge": kge,
    }

def load_prediction_table(path: Path, min_height: float, max_height: float) -> tuple[pd.DataFrame, np.ndarray, np.ndarray]:
    frame = pd.read_csv(path)
    if "aux_shot_uid" in frame.columns:
        order = [name for name in ("abs_temporal_delta_days", "sequence_center_distance", "batch_index", "candidate_order") if name in frame.columns]
        frame = frame.sort_values(order, kind="mergesort") if order else frame
        frame = frame.drop_duplicates("aux_shot_uid", keep="first").copy()
    true_col = _column(frame, ("y_true", "gedi_rh95", "rh95", "target", "observed"))
    pred_col = _column(frame, ("y_pred", "pred_on_growthloss", "prediction_original_coords", "prediction", "predicted"))
    y_true = frame[true_col].to_numpy(float)
    y_pred = frame[pred_col].to_numpy(float)
    keep = np.isfinite(y_true) & np.isfinite(y_pred) & (y_true >= min_height) & (y_true <= max_height)
    return frame.loc[keep].copy(), y_true[keep], y_pred[keep]

def scatter_panel(path: Path, forest: str, domain: tuple[float, float], output: Path) -> dict:
    _, y_true, y_pred = load_prediction_table(path, *domain)
    metrics = regression_metrics(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(7.2, 6.4), constrained_layout=True)
    hb = ax.hexbin(y_true, y_pred, gridsize=48, mincnt=1, bins="log", cmap="viridis")
    low, high = 0.0, float(domain[1])
    ax.plot([low, high], [low, high], "k--", lw=1.5, label="1:1")
    x = np.linspace(low, high, 100)
    ax.plot(x, metrics["intercept"] + metrics["slope"] * x, color="crimson", lw=1.8, label="Regression")
    ax.set(xlim=(low, high), ylim=(low, high), xlabel="Observed GEDI RH95 (m)", ylabel="Predicted height (m)")
    ax.set_title(f"{forest} — TEST unique-nearest — RH95 {domain[0]:g}–{domain[1]:g} m", fontweight="bold")
    text = "\n".join([
        f"n = {metrics['n']}", f"R² = {metrics['r2']:.4f}", f"MAE = {metrics['mae']:.4f} m",
        f"RMSE = {metrics['rmse']:.4f} m", f"Slope = {metrics['slope']:.4f}",
        f"Std ratio = {metrics['std_ratio']:.4f}", f"Bias = {metrics['bias']:+.4f} m",
        f"KGE = {metrics['kge']:.4f}",
    ])
    ax.text(0.025, 0.975, text, transform=ax.transAxes, va="top", bbox=dict(facecolor="white", alpha=.9))
    ax.legend(loc="lower right")
    fig.colorbar(hb, ax=ax, label="log10(N)")
    output.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output, dpi=600, bbox_inches="tight")
    fig.savefig(output.with_suffix(".pdf"), bbox_inches="tight")
    plt.show()
    return metrics


## Final evaluation diagnostics

The selected models are evaluated on held-out TEST support. Scatter plots, height-dependent residuals, and paired Phase 1–Phase 2 comparisons are exported for the article.

In [ ]:
FINAL_RESULTS = PROJECT / "Results" / "Final_Article_Harmonized_GEDIAnchored_NaturalP1" / "Phase2"
phase2_rows = []
for site, spec in FINAL_SPECS.items():
    prediction_path = FINAL_RESULTS / spec["forest"] / "test_unique_nearest.csv.gz"
    if not prediction_path.is_file():
        print("[MISSING AOI-MASKED PREDICTIONS]", prediction_path)
        continue
    output = FINAL_RESULTS / spec["forest"] / "scatter_test.png"
    metrics = scatter_panel(prediction_path, spec["forest"], spec["domain"], output)
    phase2_rows.append({"forest": spec["forest"], "product": spec["product"], "temporal_support": "AOI masked", "D": spec["D"], "K": spec["K"], "lambda_growth": spec["lambda_growth"], "checkpoint_sha256": spec["checkpoint_sha256"], **metrics})

phase2_metrics = pd.DataFrame(phase2_rows)
FINAL_RESULTS.mkdir(parents=True, exist_ok=True)
phase2_metrics.to_csv(FINAL_RESULTS / "phase2_final_test_metrics.csv", index=False)
phase2_metrics


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8), constrained_layout=True)
for ax, (site, spec) in zip(axes, FINAL_SPECS.items()):
    path = FINAL_RESULTS / spec["forest"] / "test_unique_nearest.csv.gz"
    if not path.is_file():
        ax.set_axis_off(); continue
    _, y_true, y_pred = load_prediction_table(path, *spec["domain"])
    residual = y_pred - y_true
    edges = np.arange(0, spec["domain"][1] + 5, 5.0)
    centers, medians, q25, q75 = [], [], [], []
    for low, high in zip(edges[:-1], edges[1:]):
        values = residual[(y_true >= low) & (y_true < high)]
        if len(values) == 0: continue
        centers.append((low + high) / 2); medians.append(np.median(values))
        q25.append(np.quantile(values, .25)); q75.append(np.quantile(values, .75))
    ax.axhline(0, color="black", ls="--", lw=1)
    ax.plot(centers, medians, color="crimson", marker="o")
    ax.fill_between(centers, q25, q75, color="crimson", alpha=.2)
    ax.set(title=spec["forest"], xlabel="GEDI RH95 (m)", ylabel="Residual (predicted - observed), m")
fig.suptitle("Height-dependent error — final TEST", fontweight="bold")
figure_path = FINAL_RESULTS / "residual_profiles_three_forests.png"
fig.savefig(figure_path, dpi=600, bbox_inches="tight")
fig.savefig(figure_path.with_suffix(".pdf"), bbox_inches="tight")
plt.show()


In [ ]:
comparison_rows = []
for site, spec in FINAL_SPECS.items():
    path = FINAL_RESULTS / spec["forest"] / "test_unique_nearest.csv.gz"
    if not path.is_file(): continue
    frame = pd.read_csv(path)
    true_col = _column(frame, ("rh95", "y_true", "gedi_rh95", "target", "observed"))
    y_true = frame[true_col].to_numpy(float)
    keep = np.isfinite(y_true) & (y_true >= spec["domain"][0]) & (y_true <= spec["domain"][1])
    y_true = y_true[keep]
    phase1 = regression_metrics(y_true, frame.loc[keep, "pred_off_reference"].to_numpy(float))
    phase2 = regression_metrics(y_true, frame.loc[keep, "pred_on_growthloss"].to_numpy(float))
    comparison_rows.append({
        "forest": spec["forest"], "n": phase2["n"],
        "phase1_r2": phase1["r2"], "phase2_r2": phase2["r2"], "delta_r2": phase2["r2"] - phase1["r2"],
        "phase1_mae": phase1["mae"], "phase2_mae": phase2["mae"], "mae_gain_m": phase1["mae"] - phase2["mae"],
        "phase1_rmse": phase1["rmse"], "phase2_rmse": phase2["rmse"], "rmse_gain_m": phase1["rmse"] - phase2["rmse"],
        "phase1_slope": phase1["slope"], "phase2_slope": phase2["slope"],
        "phase1_kge": phase1["kge"], "phase2_kge": phase2["kge"],
    })
phase_comparison = pd.DataFrame(comparison_rows)
phase_comparison.to_csv(FINAL_RESULTS / "phase1_vs_phase2_same_test_metrics.csv", index=False)
phase_comparison


## Dense annual-map inference hand-off

Wall-to-wall map coverage is produced separately from the sparse GEDI training catalogue.

In [ ]:
# AGADIR_SUPPORT_ACCOUNTING_V1
from pathlib import Path as _SupportPath

_agadir_catalog = Path(r"C:\Users\Dell\Desktop\Publication_Clarck") / "Data" / "Sparse" / "Agadir" / "Catalogs" / "final_catalog" / "shot_catalog_step05.csv.gz"
_agadir_prediction = FINAL_RESULTS / "Agadir" / "test_unique_nearest.csv.gz"

_registry = pd.read_csv(_agadir_catalog, low_memory=False)
_registry = _registry[_registry["split"].astype(str).str.lower().eq("test")].copy()
_registry["rh95"] = pd.to_numeric(_registry["rh95"], errors="coerce")
_registry = _registry[_registry["rh95"].between(2.0, 20.0)].copy()
assert "aux_shot_uid" in _registry.columns
_canonical_ids = frozenset(_registry["aux_shot_uid"].dropna().astype(str))

_phase2 = pd.read_csv(_agadir_prediction, low_memory=False)
assert "aux_shot_uid" in _phase2.columns
_true_col = _column(_phase2, ("y_true", "gedi_rh95", "rh95", "target", "observed"))
_pred_col = _column(_phase2, ("y_pred", "pred_on_growthloss", "prediction_original_coords", "prediction", "predicted"))
_phase2[_true_col] = pd.to_numeric(_phase2[_true_col], errors="coerce")
_phase2[_pred_col] = pd.to_numeric(_phase2[_pred_col], errors="coerce")
_phase2 = _phase2[
    _phase2[_true_col].between(2.0, 20.0)
    & np.isfinite(_phase2[_pred_col])
].copy()
_phase2_ids = frozenset(_phase2["aux_shot_uid"].dropna().astype(str))

assert _phase2_ids <= _canonical_ids, (
    "Phase 2 support must be a subset of the canonical TEST registry.",
    len(_phase2_ids - _canonical_ids),
)
_support_audit = pd.DataFrame([
    {
        "support": "Raw canonical TEST registry",
        "definition": "unique GEDI shots; split=test; RH95 2–20 m",
        "n": len(_canonical_ids),
    },
    {
        "support": "Valid T4 Phase 2 evaluation",
        "definition": "canonical shots with a finite unique-nearest Phase 2 prediction",
        "n": len(_phase2_ids),
    },
    {
        "support": "Excluded before Phase 2 scoring",
        "definition": "canonical shots without a valid T4 Phase 2 prediction",
        "n": len(_canonical_ids - _phase2_ids),
    },
])
display(_support_audit)
assert len(_canonical_ids) == 9206, len(_canonical_ids)
assert len(_phase2_ids) == 6125, len(_phase2_ids)
print("[PASS] Agadir support labels: canonical registry n=9,206; valid T4 Phase 2 n=6,125.")
